# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aadirwt/Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule:
Prioritize webpages with high search volume, high impressions, low CTR, and average positions close to the first page of search results. These pages have the greatest potential for improvement through SEO optimization.

Reason Codes:
RC1 – High search volume
RC2 – High impressions but low CTR
RC3 – Average position between 11 and 20
RC4 – High engagement potential
RC5 – Low optimization priority

In [14]:
!git clone https://github.com/Aadirwt/Internship.git

fatal: destination path 'Internship' already exists and is not an empty directory.


In [15]:
import pandas as pd

# Load dataset
df = pd.read_csv("/content/Internship/data/raw/content_refresh_anonymized.csv")

# Display basic information
print("Dataset Shape:", df.shape)

print("\nSelected Features:")
print(df[[
    "search_volume",
    "impressions_90d",
    "ctr",
    "avg_position",
    "engagement_rate"
]].head())


Dataset Shape: (30000, 44)

Selected Features:
   search_volume  impressions_90d   ctr  avg_position  engagement_rate
0           10.0             3803  0.76          10.6             5.88
1           90.0            15320  0.05          20.3             0.00
2            0.0            12581  0.09          36.5             0.00
3           10.0            11751  0.49           6.2             1.28
4            0.0            19140  0.13          44.0             0.00


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

A baseline action score is calculated using key SEO metrics. Pages with higher search volume and impressions receive higher scores, while pages with low CTR and positions near the first search results page are prioritized for optimization. The ranked queue identifies the webpages that should be reviewed first.

In [16]:
import pandas as pd

# Load dataset
df = pd.read_csv("/content/Internship/data/raw/content_refresh_anonymized.csv")

# Normalize features
df["sv_norm"] = df["search_volume"] / df["search_volume"].max()
df["imp_norm"] = df["impressions_90d"] / df["impressions_90d"].max()
df["ctr_inv"] = 1 - df["ctr"]
df["pos_score"] = 1 / (df["avg_position"] + 1)

# Baseline Action Score
df["baseline_action_score"] = (
    0.35 * df["sv_norm"] +
    0.30 * df["imp_norm"] +
    0.20 * df["ctr_inv"] +
    0.15 * df["pos_score"]
)

# Rank pages
ranked = df.sort_values(
    by="baseline_action_score",
    ascending=False
)

# Create output folder
import os
os.makedirs("/content/Internship/work/outputs", exist_ok=True)

# Save CSV
ranked.to_csv(
    "/content/Internship/work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV created successfully.")
print(ranked[[
    "content_id",
    "baseline_action_score"
]].head(10))


CSV created successfully.
                 content_id  baseline_action_score
12140  content_ef99c4abd9ab               0.549545
6653   content_5fe46e04994d               0.509833
26844  content_8c19996aa890               0.508284
17812  content_aaef01a50def               0.493897
18701  content_deb54e9e19cd               0.492304
28282  content_454cc6654c6e               0.491945
6972   content_bf67a444faef               0.491195
17907  content_5ec29ae79c60               0.490515
19636  content_2cb567c3c89b               0.474883
22788  content_ee4630879d03               0.443604


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The ranked results were reviewed to ensure that the prioritization rule produced reasonable recommendations. High-priority pages generally combined strong search demand with opportunities for improved click-through rates or rankings. The review also considered possible limitations such as seasonal trends, incomplete engagement data, and factors outside the available dataset that could influence performance.

In [17]:
import pandas as pd

top20 = pd.read_csv(
    "/content/Internship/work/outputs/baseline_action_score.csv"
).head(20)

print(top20[[
    "content_id",
    "baseline_action_score",
    "search_volume",
    "impressions_90d",
    "ctr",
    "avg_position"
]])

              content_id  baseline_action_score  search_volume  \
0   content_ef99c4abd9ab               0.549545        74000.0   
1   content_5fe46e04994d               0.509833         1900.0   
2   content_8c19996aa890               0.508284           70.0   
3   content_aaef01a50def               0.493897         4400.0   
4   content_deb54e9e19cd               0.492304        60500.0   
5   content_454cc6654c6e               0.491945        60500.0   
6   content_bf67a444faef               0.491195        60500.0   
7   content_5ec29ae79c60               0.490515        60500.0   
8   content_2cb567c3c89b               0.474883            0.0   
9   content_ee4630879d03               0.443604        49500.0   
10  content_cd6760921db8               0.437800        49500.0   
11  content_83e3da1394ac               0.437203        49500.0   
12  content_004d8a5ce838               0.435609        18100.0   
13  content_4c36c775b818               0.431998           40.0   
14  conten

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [18]:
import pandas as pd
import os

for root, dirs, files in os.walk("/content/Internship"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/content/Internship/data/raw/content_refresh_anonymized.csv
/content/Internship/outputs/refresh_queue_sample.csv
/content/Internship/work/outputs/baseline_action_score.csv


In [19]:
import pandas as pd

df = pd.read_csv("/content/Internship/data/raw/content_refresh_anonymized.csv")

print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


Some lower-ranked pages may still become important because of seasonal demand or future trends that are not captured in the current dataset. No future information or label-derived fields were included in the scoring rule, reducing the possibility of data leakage. The baseline rule relies only on information available at the decision time.

In [20]:
import pandas as pd

df = pd.read_csv("/content/Internship/data/raw/content_refresh_anonymized.csv")

excluded = [
    "trend_direction",
    "trend_pct",
    "clicks_last_30d",
    "sessions_last_30d"
]

print("Excluded / Future-sensitive fields:")
for col in excluded:
    print("-", col)

print("\nNo future labels were used in the baseline score.")
print("Baseline uses only current observable SEO metrics.")


Excluded / Future-sensitive fields:
- trend_direction
- trend_pct
- clicks_last_30d
- sessions_last_30d

No future labels were used in the baseline score.
Baseline uses only current observable SEO metrics.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.